# Setup

In [ ]:
import os
import sys

sys.path.insert(0, ".")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import common

PLOTS_DIR = common.RESULTS_DIR / "plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)


def show(name):
    """Export the current figure as PDF for LaTeX, then display it."""
    plt.gcf().savefig(PLOTS_DIR / f"{name}.pdf", bbox_inches="tight")
    plt.show()

b1 = pd.read_csv(common.SOURCE_DIR / "Building_1.csv")
weather = pd.read_csv(common.SOURCE_DIR / "weather.csv")
PV_KW = 4.0  # schema: Building_1 pv nominal_power. The CSV stores W per kW installed.

series = pd.DataFrame({
    "load": b1["non_shiftable_load"],                  # kWh per hour
    "solar": b1["solar_generation"] * PV_KW / 1000.0,  # kWh per hour, CityLearn's scaling
    "price": pd.read_csv(common.SOURCE_DIR / "pricing.csv")["electricity_pricing"],
    "temperature": weather["outdoor_dry_bulb_temperature"],
    "humidity": weather["outdoor_relative_humidity"],
    "diffuse": weather["diffuse_solar_irradiance"],
    "direct": weather["direct_solar_irradiance"],
})
series["hour"] = b1["hour"]
series["net"] = series["load"] - series["solar"]  
T = len(series)

series[["load", "solar", "price"]].describe().round(3)

Das Haus zieht 1.2 kWh / Stunde, solar produziert im median nichts aber 3.9 kwh in der Spitze, Preis ist nahezu immer sehr niedrig / am unteren Ende der Preisrange

## example day

In [ ]:
prof = series.groupby("hour")[["load", "solar"]].mean()
by_hour = series.groupby("hour")["price"].agg(["min", "mean", "max"])
fig, (ax, ax2) = plt.subplots(1, 2, figsize=(10, 3.5))
ax.plot(prof.index, prof["load"], label="load")
ax.plot(prof.index, prof["solar"], label="solar")
ax.set(xlabel="hour of day", ylabel="kWh/h", title="average day")
ax.legend()
ax2.plot(by_hour.index, by_hour["mean"], label="mean")
ax2.fill_between(by_hour.index, by_hour["min"], by_hour["max"], alpha=0.25,
                 label="min/max")
ax2.set(xlabel="hour of day", ylabel="$/kWh", title="price by hour of day")
ax2.legend()
fig.tight_layout()
show("average_day_and_price")

surplus = np.clip(series["solar"] - series["load"], 0, None).to_numpy()
daily_surplus = surplus.reshape(-1, 24).sum(axis=1)
fig, ax = plt.subplots(figsize=(5.5, 3.5))
ax.hist(daily_surplus, bins=40)
ax.axvline(6.4, color="k", ls="--", label="battery capacity")
ax.set(xlabel="PV surplus per day [kWh]", title="solar with nowhere to go")
ax.legend()
show("daily_surplus")

print(f"annual load     {series['load'].sum():7.0f} kWh")
print(f"annual solar    {series['solar'].sum():7.0f} kWh")
print(f"annual surplus  {surplus.sum():7.0f} kWh "
      f"({surplus.sum() / series['solar'].sum():.0%} of PV output, currently wasted)")
print(f"days where surplus exceeds the 6.4 kWh battery: {(daily_surplus > 6.4).mean():.0%}")

Solar peaks mittags, während Building Energie primär abends braucht. ca Häfte von Solar wird nicht genutzt. 

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 3))
ax.plot(series["price"].iloc[:336].to_numpy())
ax.set(xlabel="hour", ylabel="$/kWh", title="two weeks of the tariff")
show("tariff_two_weeks")


## forecasts 

In [ ]:
from forecast_chronos import FORECAST_CSV, FC_SERIES, HORIZONS, build_table, shifted_paths

chronos_fc = pd.read_csv(FORECAST_CSV, index_col=0)
print(f"{FORECAST_CSV.name}: {chronos_fc.shape[0]} origins x {chronos_fc.shape[1]} features")
chronos_fc.head(3)

In [ ]:
oracle_fc = build_table(shifted_paths(series, "oracle"))
naive_fc = build_table(shifted_paths(series, "naive"))

valid = slice(48, T - 24)
rows = []
for col in chronos_fc.columns:
    truth = oracle_fc[col].to_numpy()[valid]
    for model, fc in [("chronos", chronos_fc), ("naive", naive_fc)]:
        err = fc[col].to_numpy()[valid] - truth
        rows.append({"feature": col, "model": model, "mae": np.abs(err).mean()})
mae = pd.DataFrame(rows).pivot(index="feature", columns="model", values="mae")
mae["chronos_vs_naive"] = mae["chronos"] / mae["naive"] - 1  # <0: chronos better
print(mae.reindex(chronos_fc.columns).round(3).to_string())

model_colors = {"naive": "#bbbbbb", "chronos": "#D55E00"}
panels = [("load", "h", "load, points [kWh]"),
          ("load", "sum", "load, volumes [kWh]"),
          ("solar", "h", "solar, points [kWh]"),
          ("solar", "sum", "solar, volumes [kWh]")]
fig, axes = plt.subplots(2, 2, figsize=(10, 5.5))
for ax, (series_name, kind, title) in zip(axes.flat, panels):
    cols = [f"{series_name}_{kind}{h}" for h in HORIZONS]
    x = np.arange(len(HORIZONS))
    for k, model in enumerate(["naive", "chronos"]):
        vals = mae.loc[cols, model].to_numpy()
        ax.bar(x + (k - 0.5) * 0.36, vals, width=0.34,
               color=model_colors[model], label=model)
    for i, col in enumerate(cols):
        rel = mae.loc[col, "chronos_vs_naive"]
        ax.text(i + 0.18, mae.loc[col, "chronos"], f" {rel:+.0%}",
                ha="center", va="bottom", fontsize=8,
                color="#B22222" if rel > 0 else "#006400")
    ax.set_xticks(x, [f"{h} h" for h in HORIZONS])
    ax.set_title(title, fontsize=10)
    ax.set_ylabel("mean abs. error")
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.suptitle("forecast error: chronos vs same-hour-yesterday", y=1.02)
fig.legend(handles, labels, loc="upper right", ncol=2, frameon=False,
           bbox_to_anchor=(0.98, 1.03))
fig.tight_layout()
show("forecast_mae_chronos_vs_naive")

In [ ]:
s0, e0 = common.EVAL_BLOCKS[0]
week = np.arange(s0, e0 + 1)

def as_days(t):
    return (t - s0) / 24


for fc, color, label, slug in [
    (chronos_fc, "#D55E00", "Chronos-2", "chronos"),
    (naive_fc, "#009E73", "same hour yesterday", "naive"),
]:
    fig, ax = plt.subplots(figsize=(6, 2.6))
    ax.plot(as_days(week), oracle_fc["load_h6"].iloc[week],
            color="#c8c8c8", lw=1.1, label="actual load")
    ax.plot(as_days(week), fc["load_h6"].iloc[week], color=color, lw=1.4, label=label)
    ax.set(title=f"point feature: household load 6 hours ahead ({label})",
           xlabel="day of the held-out week",
           ylabel="load in 6 h [kWh]")
    ax.legend(loc="upper left", fontsize=8, ncol=2)
    show(f"week_load_h6_{slug}")


truth_v = oracle_fc["load_sum24"].iloc[week]
chronos_v = chronos_fc["load_sum24"].iloc[week]
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(as_days(week), truth_v, color="#444444", lw=2, label="actual energy")
ax.plot(as_days(week), chronos_v, color="#D55E00", lw=1.2, label="Chronos-2")
ax.plot(as_days(week), naive_fc["load_sum24"].iloc[week],
        color="#009E73", lw=1, ls="--", label="same hour yesterday")
ax.fill_between(as_days(week), chronos_v, truth_v,
                color="#D55E00", alpha=0.15, label="Chronos underestimation")
ax.set(title="volume feature: household energy over the next 24 hours",
       xlabel="day of the held-out week",
       ylabel="load over next 24 h [kWh]")
ax.legend(fontsize=8)
show("week_load_sum24")

## control experiment

In [ ]:
from forecast_train import ARMS, SEEDS, evaluate_one

for arm, (source, cols) in ARMS.items():
    src = f" from {source}" if source else ""
    print(f"{arm:15s} {len(cols):2d} forecast features{src}")

In [ ]:
trained = [(a, s) for a in ARMS for s in SEEDS
           if (common.MODELS_DIR / f"sac_b1_{a}_s{s}.pt").exists()]
missing = [f"{a}_s{s}" for a in ARMS for s in SEEDS if (a, s) not in trained]
print(f"{len(trained)}/{len(ARMS) * len(SEEDS)} runs trained"
      + (f", missing: {', '.join(missing)}" if missing else ""))

In [ ]:
KPIS_CSV = common.RESULTS_DIR / "forecast_kpis.csv"
results = (pd.read_csv(KPIS_CSV) if KPIS_CSV.exists()
           else pd.DataFrame(columns=["arm", "seed", "score"]))

new_rows = []
for arm in ARMS:
    for seed in SEEDS:
        if not (common.MODELS_DIR / f"sac_b1_{arm}_s{seed}.pt").exists():
            continue
        if ((results["arm"] == arm) & (results["seed"] == seed)).any():
            continue  # already evaluated
        new_rows.append({"arm": arm, "seed": seed, "score": evaluate_one(arm, seed)})
        print(new_rows[-1], flush=True)

if new_rows:
    results = pd.concat([results, pd.DataFrame(new_rows)], ignore_index=True)
    common.RESULTS_DIR.mkdir(exist_ok=True)
    results.to_csv(KPIS_CSV, index=False)
print(f"{len(results)} evaluated runs in {KPIS_CSV.name}")

## Results

In [ ]:
if results.empty:
    print("nothing trained yet; this section fills in after the cluster runs")
else:
    summary = (results[results["arm"].isin(ARMS)]
               .groupby("arm")["score"].agg(["mean", "std", "count"])
               .reindex(list(ARMS)))
    oracle_mean = summary.loc["all_oracle", "mean"]
    ceiling = oracle_mean
    ceiling_label = ("all_oracle (learned ceiling)" if pd.notna(oracle_mean)
                     else "LP replay (stand-in ceiling)")
    summary["gap_closed"] = (1 - summary["mean"]) / (1 - ceiling)
    print(summary.round(3).to_string())

    if summary["mean"].notna().sum() == 0:
        print("\nno runs for the current arm set yet")
    else:
        plot_order = ["baseline", "price", "weather",
                      "seasonal_naive_points", "demand_points",
                      "seasonal_naive_volumes", "demand_volumes",
                      "all", "all_oracle"]
        plot_order = [a for a in plot_order if pd.notna(summary.loc[a, "mean"])]

        fig, ax = plt.subplots(figsize=(8, 3.6))
        ys = np.arange(len(plot_order))[::-1] 
        for y, arm in zip(ys, plot_order):
            seeds_scores = results.loc[results["arm"] == arm, "score"]
            ax.scatter(seeds_scores, [y] * len(seeds_scores),
                       facecolors="none", edgecolors="#999999", s=30, zorder=2,
                       label="single seed" if y == ys[0] else None)
            ax.scatter(summary.loc[arm, "mean"], y, color="#222222", s=55, zorder=3,
                       label="mean over seeds" if y == ys[0] else None)

        ax.axvline(summary.loc["baseline", "mean"], ls="--", c="#E69F00",
                   lw=1.4, label="baseline (no forecast)")
        ax.axvline(ceiling, ls=":", c="#0072B2", lw=1.4, label=ceiling_label)
        ax.set_yticks(ys)
        ax.set_yticklabels(plot_order)
        ax.xaxis.set_major_locator(plt.MultipleLocator(0.005))
        ax.xaxis.set_major_formatter(plt.FormatStrFormatter("%.3f"))
        ax.set_xlabel("electricity cost relative to no battery")
        ax.set_title("Held-out weeks: cost per forecast-feature set, by seed",
                     fontsize=11)
        ax.grid(axis="x", alpha=0.3)
        ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1.0),
                  fontsize=8, frameon=False)
        plt.tight_layout()
        show("arm_scores_by_seed")

In [ ]:
if not results.empty:
    pairs = [("points", "seasonal_naive_points", "demand_points"),
             ("volumes", "seasonal_naive_volumes", "demand_volumes")]
    model_colors = {"seasonal naive": "#009E73", "chronos": "#D55E00"}

    fig, ax = plt.subplots(figsize=(7.5, 2.6))
    ys = np.arange(len(pairs))[::-1]
    for y, (rep, naive_arm, chronos_arm) in zip(ys, pairs):
        ax.plot([summary.loc[naive_arm, "mean"], summary.loc[chronos_arm, "mean"]],
                [y, y], color="#bbbbbb", lw=2, zorder=1)
        for arm, model in [(naive_arm, "seasonal naive"), (chronos_arm, "chronos")]:
            c = model_colors[model]
            seeds_scores = results.loc[results["arm"] == arm, "score"]
            ax.scatter(seeds_scores, [y] * len(seeds_scores),
                       facecolors="none", edgecolors=c, s=30, zorder=2,
                       label=f"{model}, single seed" if y == ys[0] else None)
            ax.scatter(summary.loc[arm, "mean"], y, color=c, s=60, zorder=3,
                       label=f"{model}, mean" if y == ys[0] else None)

    ax.set_yticks(ys)
    ax.set_yticklabels([f"{rep}\n(load + solar at 6/12/24 h)" for rep, *_ in pairs])
    ax.xaxis.set_major_locator(plt.MultipleLocator(0.005))
    ax.xaxis.set_major_formatter(plt.FormatStrFormatter("%.3f"))
    ax.set_xlabel("electricity cost relative to no battery (lower is better)")
    ax.set_title("Chronos-2 vs. same-hour-yesterday", fontsize=11)
    ax.set_ylim(-0.5, len(pairs) - 0.5)
    ax.grid(axis="x", alpha=0.3)
    ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1.0), fontsize=8, frameon=False)
    plt.tight_layout()
    show("chronos_vs_naive_control")

In [ ]:
groups = [("load", "h", "load, points"), ("solar", "h", "solar, points"),
          ("load", "sum", "load, volumes"), ("solar", "sum", "solar, volumes")]
labels, vals = [], []
for s, kind, title in groups:
    for h in HORIZONS:
        labels.append(f"{title}, {h} h")
        vals.append(mae.loc[f"{s}_{kind}{h}", "chronos_vs_naive"])
vals = np.array(vals) * 100

fig, ax = plt.subplots(figsize=(7.5, 4.2))
ys = np.arange(len(labels))[::-1]
bar_colors = ["#0072B2" if v < 0 else "#D55E00" for v in vals]
ax.barh(ys, vals, color=bar_colors, height=0.6)
for y, v in zip(ys, vals):
    ax.text(v + (0.5 if v >= 0 else -0.5), y, f"{v:+.0f}%",
            va="center", ha="left" if v >= 0 else "right", fontsize=8)
ax.axvline(0, color="#444444", lw=1)
ax.set_yticks(ys)
ax.set_yticklabels(labels, fontsize=9)
for i in (3, 6, 9):  
    ax.axhline(ys[i] + 0.5, color="#dddddd", lw=0.8)
ax.set_xlabel("Chronos-2 forecast error relative to same-hour-yesterday"
              " (negative = Chronos more accurate)")
ax.set_title("Forecast accuracy by horizon: Chronos-2 vs seasonal naive",
             fontsize=11)
lim = max(abs(vals)) * 1.25
ax.set_xlim(-lim, lim)
plt.tight_layout()
show("forecast_accuracy_by_horizon")

## Shaping term 

In [ ]:
ablation_files = [
    (common.RESULTS_DIR / "learning_curve_b1_demand_oracle_noshaping_s0.csv",
     "difference reward only"),
    (common.RESULTS_DIR / "learning_curve_b1_demand_oracle_s0.csv",
     "+ potential shaping"),
]
if all(f.exists() for f, _ in ablation_files):
    fig, ax = plt.subplots(figsize=(7.5, 3))
    for f, label in ablation_files:
        curve = pd.read_csv(f)
        ax.plot(curve["timestep"],
                curve["episode_return"].rolling(5, min_periods=1).mean(), label=label)
    ax.axhline(0, color="grey", lw=0.8)
    ax.set(xlabel="environment step", ylabel="train episode return",
           title="same features (oracle demand), same seed, only the reward differs")
    ax.legend()
    show("reward_shaping_ablation")
else:
    print("ablation curves not found")